<a href="https://colab.research.google.com/github/sunbal822/Ai-recipe-rescue/blob/main/AI-Recipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.0 MB/s eta 0:00:00


In [18]:
import re

# splits the LLM's labeled response into a dict, e.g. {"RECIPE": "...", "STEPS": "..."}
def parse_recipe(text):
    labels = ["RECIPE", "PRIORITY NOTE", "EXTRA INGREDIENTS NEEDED", "COOK TIME", "STEPS", "REALISM CHECK"]
    pattern = r"(" + "|".join(labels) + r"):"
    parts = re.split(pattern, text)
    result = {label: "" for label in labels}
    # parts alternates [junk, label, content, label, content, ...]
    for i in range(1, len(parts) - 1, 2):
        label = parts[i].strip()
        if label in result:
            result[label] = parts[i + 1].strip()
    return result


# force_final=True skips the clarifying-question option (used after user answers)
def get_recipe_step(ingredients, temperature=0.7, force_final=False):
    if not ingredients or not ingredients.strip():
        return "EMPTY", "Please enter at least one ingredient to get a recipe."

    clarify_block = "" if force_final else """
Before giving a recipe, think about what KEY ingredients (proteins, vegetables, starches/grains, bread/flour, dairy) are missing to make a good dish — do NOT assume the user has any vegetables, starches, or bread/flour just because basic staples like salt/oil/water are allowed.
If one or more such key ingredients are missing and asking would meaningfully improve the recipe, ask ONE combined question covering ALL of them at once (not multiple separate questions).
Example: "Do you have any vegetables like onion or celery, and something like flour or bread?"
To do that, respond with ONLY this line: CLARIFY: <your one combined question>
Only skip straight to the recipe if the ingredient list already has enough variety (e.g. a protein + a vegetable + a starch) that no reasonable clarification is needed.
"""

    prompt = f"""You are a practical home cooking assistant. A user will give you
a messy, possibly typo-filled, informal list of ingredients they have at home.

User's ingredients: {ingredients}

Interpret the list even if it's vague, has typos, or very few items.
{clarify_block}
STRICT RULE: Only use ingredients the user has actually stated. Assume ONLY basic staples (salt, oil, water, pepper, butter). Do NOT silently add any other ingredient (no garlic, lemon, herbs, spices, zest, etc.) into the recipe steps unless the user listed it.
If the recipe would genuinely benefit from something they didn't mention, put it in EXTRA INGREDIENTS NEEDED instead of using it directly in the steps. If the user has said they have no more ingredients, work ONLY with what they've confirmed, even if the recipe is simple as a result.

INSUFFICIENT CHECK: If, even after any clarification, what the user has is NOT enough to form a real main dish (e.g. only a seasoning, garnish, or single condiment like salt, pepper, lemon, herbs alone, with no protein/vegetable/starch/grain base), do NOT invent a fake main dish recipe. Instead respond with ONLY this line:
INSUFFICIENT: <one honest sentence saying this isn't enough for a full meal> | <a small side, garnish, or condiment idea achievable with just this, if any, else say "no small idea either">

When giving a full recipe (only if ingredients ARE sufficient), respond with a single realistic recipe, formatted EXACTLY like this:

RECIPE: <name>
PRIORITY NOTE: <briefly mention if you prioritized any perishable items like eggs/spinach over pantry staples>
EXTRA INGREDIENTS NEEDED: <list any, or "None" if not needed. If something is missing, suggest a common substitution instead when reasonable>
COOK TIME: <estimated total time>
STEPS:
1. ...
2. ...
REALISM CHECK: <one honest sentence — is this a real dish or more of a resourceful hack?>

Assume basic staples (salt, oil, water, pepper, butter) are always available.
Keep it realistic and achievable right now."""

    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
        )
        text = response.choices[0].message.content.strip()
    except Exception as e:
        return "ERROR", f"Something went wrong getting your recipe: {e}"

    # model chose to ask instead of answering
    if text.startswith("CLARIFY:"):
        return "CLARIFY", text[len("CLARIFY:"):].strip()

    # model decided ingredients can't form a real meal
    if text.startswith("INSUFFICIENT:"):
        return "INSUFFICIENT", text[len("INSUFFICIENT:"):].strip()

    return "RECIPE", text


# quick terminal test
print(get_recipe_step("chicken"))

('CLARIFY', 'Do you have any vegetables (e.g., onion, carrot, bell pepper), a starch or grain (e.g., rice, potatoes, pasta), and any dairy (e.g., cheese, milk, butter)?')


In [5]:
!pip install gradio -q

In [ ]:
import gradio as gr

In [ ]:
import gradio as gr

def handle_submit(ingredients, chat_state):
    status, content = get_recipe_step(ingredients, temperature=0.7)

    if status == "CLARIFY":
        chat_state = {"original": ingredients, "question": content}
        return (chat_state, gr.update(value=content, visible=True),
                "", "", "", "", "", gr.update(visible=True))

    if status == "INSUFFICIENT":
        # content is "<reason> | <small idea>" — split and show plainly, no fake recipe
        reason, _, idea = content.partition("|")
        return (chat_state, gr.update(value="", visible=False),
                "⚠️ Not enough for a full recipe", reason.strip(),
                "", idea.strip(), "", gr.update(visible=False))

    parsed = parse_recipe(content) if status == "RECIPE" else {}
    error_msg = content if status != "RECIPE" else ""
    return (chat_state, gr.update(value="", visible=False),
            parsed.get("RECIPE", error_msg), parsed.get("PRIORITY NOTE", ""),
            parsed.get("EXTRA INGREDIENTS NEEDED", ""),
            f"{parsed.get('COOK TIME','')}\n\n{parsed.get('STEPS','')}",
            parsed.get("REALISM CHECK", ""), gr.update(visible=False))


def handle_answer(answer, chat_state):
    combined = f"{chat_state.get('original','')}. Additional info: {answer}"
    status, content = get_recipe_step(combined, temperature=0.7, force_final=True)

    if status == "INSUFFICIENT":
        reason, _, idea = content.partition("|")
        return (gr.update(value="", visible=False),
                "⚠️ Not enough for a full recipe", reason.strip(),
                "", idea.strip(), "", gr.update(visible=False))

    parsed = parse_recipe(content) if status == "RECIPE" else {}
    error_msg = content if status != "RECIPE" else ""
    return (gr.update(value="", visible=False),
            parsed.get("RECIPE", error_msg), parsed.get("PRIORITY NOTE", ""),
            parsed.get("EXTRA INGREDIENTS NEEDED", ""),
            f"{parsed.get('COOK TIME','')}\n\n{parsed.get('STEPS','')}",
            parsed.get("REALISM CHECK", ""), gr.update(visible=False))


def handle_surprise(ingredients):
    status, content = get_recipe_step(ingredients, temperature=1.1, force_final=True)

    if status == "INSUFFICIENT":
        reason, _, idea = content.partition("|")
        return ("⚠️ Not enough for a full recipe", reason.strip(), "", idea.strip(), "")

    parsed = parse_recipe(content) if status == "RECIPE" else {}
    error_msg = content if status != "RECIPE" else ""
    return (parsed.get("RECIPE", error_msg), parsed.get("PRIORITY NOTE", ""),
            parsed.get("EXTRA INGREDIENTS NEEDED", ""),
            f"{parsed.get('COOK TIME','')}\n\n{parsed.get('STEPS','')}",
            parsed.get("REALISM CHECK", ""))

custom_css = """
.gradio-container { max-width: 700px !important; margin: auto !important; font-family: 'Segoe UI', sans-serif; }
#title { text-align: center; margin-bottom: 0px; }
#subtitle { text-align: center; color: #666; margin-top: 0px; margin-bottom: 20px; }
textarea { font-family: 'Courier New', monospace; font-size: 14px; }
#recipe_box textarea { background-color: #fff4e6; font-weight: bold; font-size: 18px; }
#priority_box textarea { background-color: #f0fff4; }
#extra_box textarea { background-color: #fff0f0; }
#steps_box textarea { background-color: #fffaf0; }
"""

with gr.Blocks(css=custom_css) as demo:
    # NOTE: state must be created INSIDE the Blocks context, not outside
    state = gr.State({})

    gr.Markdown("# 🍳 AI Recipe Rescue", elem_id="title")
    gr.Markdown("Type in whatever ingredients you have at home, messy input is fine.", elem_id="subtitle")

    ingredients_input = gr.Textbox(label="Your ingredients", placeholder="e.g. chicken")

    with gr.Row():
        submit_btn = gr.Button("Get Recipe", variant="primary")
        surprise_btn = gr.Button("🎲 Surprise Me")

    question_box = gr.Textbox(label="🤔 Quick question", visible=False, interactive=False)
    answer_input = gr.Textbox(label="Your answer", visible=False, placeholder="e.g. yes, I have rice")
    answer_btn = gr.Button("Answer", visible=False)

    recipe_box = gr.Textbox(label="Recipe", elem_id="recipe_box")
    priority_box = gr.Textbox(label="Priority Note", elem_id="priority_box")
    extra_box = gr.Textbox(label="Extra Ingredients Needed", elem_id="extra_box")
    steps_box = gr.Textbox(label="Cook Time & Steps", lines=10, elem_id="steps_box")
    realism_box = gr.Textbox(label="Realism Check")

    submit_btn.click(
        fn=handle_submit, inputs=[ingredients_input, state],
        outputs=[state, question_box, recipe_box, priority_box, extra_box, steps_box, realism_box, answer_input]
    )

    question_box.change(fn=lambda q: (gr.update(visible=bool(q)), gr.update(visible=bool(q))),
                         inputs=question_box, outputs=[answer_input, answer_btn])

    answer_btn.click(
        fn=handle_answer, inputs=[answer_input, state],
        outputs=[question_box, recipe_box, priority_box, extra_box, steps_box, realism_box, answer_input]
    )

    surprise_btn.click(
        fn=handle_surprise, inputs=ingredients_input,
        outputs=[recipe_box, priority_box, extra_box, steps_box, realism_box]
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_1551/2537703580.py:71: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e3dbf8aefea4d97206.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
